# Demo 16 - Hypothesis hunt: living-off-the-land binary (LOLBin) abuse

**Pool:** Medium · **Visual:** usage bars + suspicious-execution timeline

**The question:** are trusted Windows binaries being used to do untrustworthy things?

Attackers prefer tools that are already installed, already signed, and unlikely to be
blocked - certutil, rundll32, mshta and friends. The binaries are entirely legitimate. The
arguments are what give it away.

This notebook pairs a curated list of those binaries with a set of suspicious argument
patterns, scores every execution and ranks the result. A hunt is a hypothesis plus a scoring
rule, and here both live in an editable cell so you can iterate as the hypothesis changes.

## 1. Connect to the data lake

`MicrosoftSentinelProvider` is the bridge between this notebook and your lake. The `spark`
session is handed to you by the Microsoft Sentinel kernel, so never create your own.

Nothing is read yet. This cell only opens the connection.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F
data_provider = MicrosoftSentinelProvider(spark)

## 2. Settings you can change - the hypothesis, written down

A hunt is a hypothesis plus a scoring rule, and this cell is both of them.

- `LOLBINS` - trusted Windows binaries that attackers abuse precisely *because* they are
  signed, present on every machine and rarely blocked. "Living off the land" is the term,
  and these are the land.
- `SUS` - a regular expression of argument patterns that make an otherwise legitimate binary
  suspicious: a URL, `-decode`, `-enc`, `downloadstring`, `frombase64` and so on.

Both are meant to be edited. Watching the results shift as you adjust the hypothesis is the
demo.

In [ ]:
WORKSPACE = "your-workspace-name"
LOOKBACK_DAYS = 14
LOLBINS = ["certutil.exe","rundll32.exe","regsvr32.exe","mshta.exe","wmic.exe",
           "bitsadmin.exe","msbuild.exe","installutil.exe","cscript.exe","wscript.exe",
           "powershell.exe","curl.exe"]
SUS = r"(http[s]?://|-decode|-urlcache|javascript:|frombase64|-enc |downloadstring|iex |\.dll,|scrobj)"

## 3. Find LOLBin executions and score them

Filter `DeviceProcessEvents` down to the binaries on the list, then flag each execution
against the suspicious-argument regex.

The output counts, per binary, how many times it ran at all versus how many of those runs
looked suspicious. `certutil.exe` running is completely normal. `certutil.exe -urlcache
-split -f http://...` is certutil being used as a file downloader, which is not.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

proc = data_provider.read_table("DeviceProcessEvents", WORKSPACE)
proc = (proc.filter(F.col("TimeGenerated") >= F.expr(f"current_timestamp() - INTERVAL {int(LOOKBACK_DAYS)} DAYS"))
            .filter(F.lower("FileName").isin(LOLBINS))
            .withColumn("suspicious", F.col("ProcessCommandLine").rlike("(?i)"+SUS).cast("int")))

usage = (proc.groupBy(F.lower("FileName").alias("lolbin"))
             .agg(F.count("*").alias("runs"), F.sum("suspicious").alias("suspicious_runs"))
             .orderBy(F.desc("suspicious_runs"))).toPandas()
usage

## 4. Compare normal usage against suspicious usage

Left chart: two bars per binary. Grey is every execution, red is the suspicious subset. The
ratio between them is the number that matters - a binary that runs constantly with a handful
of odd invocations is a very different picture from one that only ever runs oddly.

Right chart: each suspicious execution plotted over time, by binary.

**What to look for:** clusters on the right-hand chart. Several suspicious executions of
*different* binaries within the same few minutes is an attack chain, not coincidence. The
table underneath gives you the full command lines to confirm it.

In [ ]:
sus = (proc.filter(F.col("suspicious")==1)
           .select("TimeGenerated","DeviceName","FileName","ProcessCommandLine")
           .orderBy(F.desc("TimeGenerated")).limit(500)).toPandas()

if usage.empty:
    print(f"No LOLBin executions in the last {LOOKBACK_DAYS} days - raise LOOKBACK_DAYS.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    u = usage.iloc[::-1]
    ax[0].barh(u["lolbin"], u["runs"], color="#bdc3c7", label="all runs")
    ax[0].barh(u["lolbin"], u["suspicious_runs"], color="#c0392b", label="suspicious")
    ax[0].set_title("LOLBin usage vs suspicious usage"); ax[0].legend()
    if not sus.empty:
        ts = pd.to_datetime(sus["TimeGenerated"])
        lb = sus["FileName"].str.lower().astype("category")
        ax[1].scatter(ts, lb.cat.codes, c=lb.cat.codes, cmap="tab10", alpha=.7)
        ax[1].set_yticks(range(len(lb.cat.categories))); ax[1].set_yticklabels(lb.cat.categories)
        ax[1].set_title("Suspicious LOLBin executions over time")
    else:
        ax[1].set_title("No suspicious argument patterns matched")
    plt.tight_layout(); plt.show()

sus.head(20)

## Why this is a notebook hunt, not a KQL query

A hunt is a hypothesis plus scoring, not a static rule. Here we combine a curated LOLBin list with regex arg-scoring and rank the results - easy to iterate cell-by-cell as the hypothesis evolves, with the timeline updating alongside.